# CamemBERT NER - Entrainement & Test (Google Colab)

Ce notebook permet d'entrainer et tester le modele CamemBERT pour la reconnaissance d'entites nommees (NER) sur des phrases de voyage SNCF.

**Labels** : `O`, `B-DEP`, `I-DEP`, `B-DEST`, `I-DEST`

**Prerequis** : Activer le GPU dans `Runtime > Change runtime type > T4 GPU`

## 1. Installation des dependances

In [ ]:
!pip install -q transformers accelerate seqeval sentencepiece

## 2. Upload des donnees

Uploadez les 3 fichiers depuis `datasets/processed/ner_retrain/` :
- `train.json`
- `val.json`
- `test.json`

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

print("Uploadez train.json, val.json et test.json")
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f"data/{filename}")
    print(f"  -> data/{filename}")

print("\nFichiers presents :", os.listdir("data"))

## 3. Configuration

In [ ]:
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    CamembertTokenizerFast,
    CamembertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)

LABEL_LIST = ["O", "B-DEP", "I-DEP", "B-DEST", "I-DEST"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL = {idx: label for idx, label in enumerate(LABEL_LIST)}

MODEL_NAME = "camembert-base"
OUTPUT_DIR = "camembert-ner-retrain-output"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 4. Chargement des donnees

In [ ]:
with open("data/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("data/val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
with open("data/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Train : {len(train_data)} exemples")
print(f"Val   : {len(val_data)} exemples")
print(f"Test  : {len(test_data)} exemples")

# Apercu
sample = train_data[0]
print(f"\nExemple :")
print(f"  Phrase : {sample['sentence']}")
for tok, lab in zip(sample["tokens"], sample["labels"]):
    if lab != "O":
        print(f"  {tok:<20} -> {lab}")

## 5. Dataset & Tokenizer

In [ ]:
class NERDataset(Dataset):
    """Dataset NER avec alignement subword pour CamemBERT."""

    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        words = sample["tokens"]
        word_labels = sample["labels"]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        word_ids = encoding.word_ids(batch_index=0)
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                if word_id < len(word_labels):
                    aligned_labels.append(LABEL2ID.get(word_labels[word_id], 0))
                else:
                    aligned_labels.append(-100)
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long),
        }


tokenizer = CamembertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = NERDataset(train_data, tokenizer)
val_dataset = NERDataset(val_data, tokenizer)
test_dataset = NERDataset(test_data, tokenizer)

print(f"Tokenizer charge : {MODEL_NAME}")
print(f"Datasets prets : train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

## 6. Modele

In [ ]:
model = CamembertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f"Labels : {LABEL_LIST}")
print(f"Params : {sum(p.numel() for p in model.parameters()):,}")

## 7. Entrainement

In [ ]:
def compute_metrics(eval_preds):
    """Metriques NER (precision, recall, F1) par entite."""
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_seq = []
        pred_seq_filtered = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            true_seq.append(ID2LABEL[label_id])
            pred_seq_filtered.append(ID2LABEL[pred_id])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_filtered)

    from seqeval.metrics import f1_score, precision_score, recall_score
    return {
        "precision": precision_score(true_labels, pred_labels, zero_division=0),
        "recall": recall_score(true_labels, pred_labels, zero_division=0),
        "f1": f1_score(true_labels, pred_labels, zero_division=0),
    }


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Lancement de l'entrainement...")
trainer.train()

## 8. Sauvegarde du modele

In [ ]:
SAVE_DIR = "camembert-ner-retrain-final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

label_config = {
    "label_list": LABEL_LIST,
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()},
}
with open(f"{SAVE_DIR}/label_config.json", "w", encoding="utf-8") as f:
    json.dump(label_config, f, indent=2)

print(f"Modele sauvegarde dans {SAVE_DIR}/")
print("Fichiers :", os.listdir(SAVE_DIR))

## 9. Evaluation sur le jeu de test

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from collections import Counter


def predict_sample(model, tokenizer, words, device, max_length=128):
    """Prediction NER sur une liste de mots."""
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    predictions = outputs.logits.argmax(dim=-1).squeeze().cpu().tolist()
    if isinstance(predictions, int):
        predictions = [predictions]

    token_word_ids = tokenizer(
        words,
        is_split_into_words=True,
        max_length=max_length,
        truncation=True,
    ).word_ids()

    word_preds = []
    prev_word_id = None
    for idx, word_id in enumerate(token_word_ids):
        if word_id is not None and word_id != prev_word_id:
            if idx < len(predictions):
                word_preds.append(ID2LABEL[predictions[idx]])
            else:
                word_preds.append("O")
        prev_word_id = word_id

    while len(word_preds) < len(words):
        word_preds.append("O")
    return word_preds[:len(words)]


# Evaluer sur le test set
model.to(device)
model.eval()

true_labels_all = []
pred_labels_all = []

for sample in test_data:
    words = sample["tokens"]
    true_labels = sample["labels"]
    pred_labels = predict_sample(model, tokenizer, words, device)

    min_len = min(len(true_labels), len(pred_labels))
    true_labels_all.append(true_labels[:min_len])
    pred_labels_all.append(pred_labels[:min_len])

print("=" * 60)
print("  RESULTATS SUR LE JEU DE TEST")
print("=" * 60)
print()
print(classification_report(true_labels_all, pred_labels_all, zero_division=0))
print(f"F1-score  : {f1_score(true_labels_all, pred_labels_all, zero_division=0):.4f}")
print(f"Precision : {precision_score(true_labels_all, pred_labels_all, zero_division=0):.4f}")
print(f"Recall    : {recall_score(true_labels_all, pred_labels_all, zero_division=0):.4f}")

## 10. Matrice de confusion

In [ ]:
import pandas as pd

matrix = {t: {p: 0 for p in LABEL_LIST} for t in LABEL_LIST}
for true_seq, pred_seq in zip(true_labels_all, pred_labels_all):
    for t, p in zip(true_seq, pred_seq):
        if t in matrix and p in matrix[t]:
            matrix[t][p] += 1

df_cm = pd.DataFrame(matrix).T
df_cm.index.name = "Vrai \\ Pred"
print("Matrice de confusion (token-level) :")
print()
print(df_cm.to_string())

## 11. Exemples de predictions sur le test set

In [ ]:
N_EXAMPLES = 15

print(f"Exemples de predictions ({N_EXAMPLES} premiers du test set) :\n")
for i, sample in enumerate(test_data[:N_EXAMPLES]):
    words = sample["tokens"]
    true_labels = sample["labels"]
    pred_labels = predict_sample(model, tokenizer, words, device)

    print(f"[{sample.get('id', i)}] {sample.get('sentence', ' '.join(words))}")
    for w, t, p in zip(words, true_labels, pred_labels):
        marker = "OK" if t == p else "XX"
        if t != "O" or p != "O":
            print(f"  {marker}  {w:<25} vrai={t:<10} pred={p:<10}")
    print()

## 12. Inference interactive

Testez le modele avec vos propres phrases.

In [ ]:
def extract_entities(words, labels):
    """Extraction des entites DEP et DEST a partir des labels NER."""
    departure_tokens = []
    destination_tokens = []
    current_tokens = []
    current_type = None

    for word, label in zip(words, labels):
        if label.startswith("B-"):
            if current_tokens and current_type:
                if "DEP" in current_type:
                    departure_tokens = current_tokens
                elif "DEST" in current_type:
                    destination_tokens = current_tokens
            current_type = label[2:]
            current_tokens = [word]
        elif label.startswith("I-") and current_type:
            current_tokens.append(word)
        else:
            if current_tokens and current_type:
                if "DEP" in current_type:
                    departure_tokens = current_tokens
                elif "DEST" in current_type:
                    destination_tokens = current_tokens
            current_tokens = []
            current_type = None

    if current_tokens and current_type:
        if "DEP" in current_type:
            departure_tokens = current_tokens
        elif "DEST" in current_type:
            destination_tokens = current_tokens

    departure = " ".join(departure_tokens) if departure_tokens else None
    destination = " ".join(destination_tokens) if destination_tokens else None
    return {"departure": departure, "destination": destination}


def predict_sentence(sentence):
    """Pipeline complet : phrase -> entites."""
    words = sentence.split()
    if not words:
        return {"departure": None, "destination": None, "labels": []}

    labels = predict_sample(model, tokenizer, words, device)
    entities = extract_entities(words, labels)
    entities["labels"] = list(zip(words, labels))
    return entities


# --- Test sur des phrases ---
test_sentences = [
    "Je veux aller de Paris a Lyon",
    "Un billet de Marseille a Bordeaux s'il vous plait",
    "Emmene-moi de Lille a Strasbourg",
    "Prochain train de Nantes vers Rennes",
    "Quel temps fait-il demain",
    "Je voudrais partir de Toulouse pour Nice",
    "Hello I want to go to London",
]

print("=" * 60)
print("  INFERENCE")
print("=" * 60)

for sentence in test_sentences:
    result = predict_sentence(sentence)
    dep = result["departure"] or "-"
    dest = result["destination"] or "-"
    print(f"\n  Phrase      : {sentence}")
    print(f"  Depart      : {dep}")
    print(f"  Destination : {dest}")
    for word, label in result["labels"]:
        if label != "O":
            print(f"    {word:<20} [{label}]")

## 13. Tester avec votre propre phrase

In [ ]:
# Modifiez la phrase ci-dessous et executez la cellule
ma_phrase = "Je veux un billet de Montpellier a Paris"

result = predict_sentence(ma_phrase)
print(f"Phrase      : {ma_phrase}")
print(f"Depart      : {result['departure'] or '-'}")
print(f"Destination : {result['destination'] or '-'}")
print()
for word, label in result["labels"]:
    print(f"  {word:<25} {label}")

## 14. Telecharger le modele entraine

In [ ]:
import shutil

archive_name = "camembert-ner-retrain-final"
shutil.make_archive(archive_name, "zip", SAVE_DIR)

print(f"Archive creee : {archive_name}.zip")
files.download(f"{archive_name}.zip")